# M7: AlpaSim — Closed-Loop Evaluation

**Stage 8: Software-in-the-Loop Testing — AlpaSim Closed-Loop**

| Item | Detail |
|------|--------|
| Simulator | AlpaSim ([NVlabs/alpasim](https://github.com/NVlabs/alpasim), Apache-2.0) |
| Driver | Alpamayo-1.5-10B — the **same checkpoint** you ran open-loop in M6 |
| Evaluation Data | NuRec scene(s) (`PhysicalAI-Autonomous-Vehicles-NuRec`) |
| This notebook | **`ml.t3.medium` (CPU)** — downloads & visualizes genuine AlpaSim results |
| Simulation runs on | a **GPU EC2 host** (yours over SSM, or an admin reference run) — *not* here |
| Output | inline plots + the real eval video |
| Source | NVlabs/alpasim |

This module evaluates the Alpamayo driving policy in **closed-loop** simulation:
the policy drives through a reconstructed real-world scene (NuRec) and AlpaSim
measures real safety metrics — `collision_at_fault`, `collision_rear`,
`dist_to_gt_trajectory`, `offroad`.

## How M7 actually runs — and why this notebook is CPU

AlpaSim is **not** a Python library you `pip install`. It is a fleet of gRPC
microservices (renderer / driver / physics / runtime / controller) orchestrated
by **Docker Compose**, and the Alpamayo driver needs a **≥40 GB GPU**. A
SageMaker Studio notebook has no Docker daemon and can't host that. So the real
simulation runs on a **GPU EC2 host**, and this CPU notebook downloads and
visualizes the genuine results it produced. There are two ways those results get
created:

- **You run it yourself** on a GPU host the workshop admin pre-provisions for
  you, reached over **SSM** (`scripts/alpasim_ec2_setup.sh`; see
  **`docs/M7_PARTICIPANT_SSM_RUNBOOK.md`**). It uploads to
  `s3://<your-workspace>/users/<you>/m7/`, and this notebook prefers that — you
  see **your own** closed-loop result. ⚠️ That GPU host bills **~$10.5/hr**;
  tell the admin when you're done so they terminate it.
- **Admin reference run** — if you didn't run your own, this notebook falls back
  to the shared admin run at `s3://<shared>/m7-reference/`. Same genuine metrics.

**M6 → M7 link (honest):** AlpaSim does *not* replay M6's predicted-trajectory
`.npy`. It loads the **same Alpamayo-1.5-10B weights** and drives them
closed-loop. M6 = the model predicting a trajectory open-loop; M7 = the same
model driving in the loop. The shared artifact is the checkpoint, not the file.

## License Notice

**WARNING: Alpamayo 1.5 model weights are under a non-commercial license.**

The evaluation shown here drives the Alpamayo 1.5 policy, whose weights are
subject to NVIDIA's non-commercial license. Commercial use requires a separate
agreement.

- Model weights (Alpamayo 1.5): **Non-commercial** (NVIDIA license)
- AlpaSim simulator code: **Apache 2.0**
- NuRec evaluation scenes: NVIDIA AV NuRec Dataset License

By proceeding, you acknowledge these licensing constraints.

In [ ]:
# ============================================================
# Config (CPU notebook — no torch, no GPU)
# ============================================================
import os
import json
import glob
import subprocess
from pathlib import Path
import boto3

ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]
PROFILE = os.environ.get("USER_PROFILE", os.environ.get("BLUEPRINT_PROFILE", "default"))
SHARED_BUCKET = os.environ.get("SHARED_BUCKET", f"av30lab-shared-data-{ACCOUNT_ID}")
S3_BUCKET = os.environ.get("USER_BUCKET", f"av30lab-user-workspace-{ACCOUNT_ID}")

# M7 has two result sources, auto-detected below (in priority order):
#   1) YOUR OWN run  — if you ran AlpaSim on your pre-provisioned GPU host (via
#      SSM; see docs/M7_PARTICIPANT_SSM_RUNBOOK.md), it wrote to your workspace at
#      users/<profile>/m7/. We prefer that so you see YOUR closed-loop result.
#   2) admin reference — otherwise fall back to the shared admin run at
#      m7-reference/ (one run, shared by all). Either way the metrics are genuine.
USER_M7_PREFIX = f"users/{PROFILE}/m7/"
ADMIN_REF_PREFIX = "m7-reference/"
# M6 output, read only for provenance ("you ran this model open-loop in M6").
M6_PREFIX = f"users/{PROFILE}/m6/"


def _has_aggregate(bucket, prefix):
    """True only if s3://bucket/prefix has an AUTHORITATIVE aggregate result —
    aggregate/results-summary.json or aggregate/metrics_results.parquet. This is
    the SAME predicate cell 4 enforces, so a partial user upload (e.g. only
    metrics_results.txt) does NOT select the user prefix and then hard-fail; it
    falls through to the admin reference instead."""
    for obj in ("results-summary.json", "metrics_results.parquet"):
        r = subprocess.run(
            ["aws", "s3", "ls", f"s3://{bucket}/{prefix}aggregate/{obj}"],
            capture_output=True, text=True,
        )
        if r.returncode == 0 and r.stdout.strip() != "":
            return True
    return False


if _has_aggregate(S3_BUCKET, USER_M7_PREFIX):
    REFERENCE_BUCKET, REFERENCE_PREFIX = S3_BUCKET, USER_M7_PREFIX
    RESULT_SOURCE = "your own EC2 run"
else:
    REFERENCE_BUCKET, REFERENCE_PREFIX = SHARED_BUCKET, ADMIN_REF_PREFIX
    RESULT_SOURCE = "admin reference run"

LOCAL_M7 = "/tmp/m7_reference"
LOCAL_M6 = "/tmp/m7_m6_provenance"
os.makedirs(LOCAL_M7, exist_ok=True)
os.makedirs(LOCAL_M6, exist_ok=True)

print(f"Profile        : {PROFILE}")
print(f"Result source  : {RESULT_SOURCE}")
print(f"Reference eval : s3://{REFERENCE_BUCKET}/{REFERENCE_PREFIX}")
print(f"M6 provenance  : s3://{S3_BUCKET}/{M6_PREFIX}")
if RESULT_SOURCE == "admin reference run":
    print(f"  (no authoritative aggregate found at s3://{S3_BUCKET}/{USER_M7_PREFIX} —")
    print("   showing the shared admin reference. Run AlpaSim yourself to see your own.)")
print("This CPU notebook downloads and visualizes genuine AlpaSim closed-loop")
print("results — it does not run the simulation itself (that needs a GPU host).")

In [ ]:
# ============================================================
# Provenance — the model M7 evaluates is the one you ran in M6 (optional)
# ============================================================
# Read M6's manifest only to show the link. M7's evaluation uses the same
# Alpamayo checkpoint (loaded by AlpaSim on the admin GPU host), NOT this file.
m6_manifest = None
try:
    subprocess.run(
        ["aws", "s3", "cp", f"s3://{S3_BUCKET}/{M6_PREFIX}manifest.json",
         os.path.join(LOCAL_M6, "manifest.json"), "--quiet"],
        check=True, capture_output=True, text=True,
    )
    with open(os.path.join(LOCAL_M6, "manifest.json")) as f:
        m6_manifest = json.load(f)
except Exception as e:
    print(f"(M6 manifest not found — provenance only, non-fatal: {e})")

if m6_manifest:
    print(f"M6 open-loop run: model = {m6_manifest.get('model')}")
    print(f"                  modes = {m6_manifest.get('modes_run')}")
    results = m6_manifest.get("results") or []
    if results and isinstance(results, list):
        m = results[0]
        if isinstance(m, dict) and "minADE_m" in m:
            print(f"                  open-loop minADE = {m['minADE_m']} m (clip {m.get('clip_id','?')[:8]}…)")
    print("M7 now drives the SAME model closed-loop in AlpaSim (see results below).")
else:
    print("Proceeding without M6 provenance — M7 reference results stand alone.")

In [ ]:
# ============================================================
# Download the genuine AlpaSim results from S3 (yours, or the admin reference)
# ============================================================
ref_s3 = f"s3://{REFERENCE_BUCKET}/{REFERENCE_PREFIX}"
print(f"Downloading {RESULT_SOURCE} from {ref_s3} ...")
r = subprocess.run(["aws", "s3", "sync", ref_s3, LOCAL_M7, "--quiet"],
                   capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError(f"Download failed: {r.stderr}")

# The authoritative aggregate is results-summary.json (or metrics_results.parquet);
# metrics_results.txt is a nice human-readable extra.
agg_txt = os.path.join(LOCAL_M7, "aggregate", "metrics_results.txt")
have_summary = any(os.path.exists(os.path.join(LOCAL_M7, "aggregate", f))
                   for f in ("results-summary.json", "metrics_results.parquet"))
if not have_summary:
    raise RuntimeError(
        "No M7 AlpaSim results found in S3. Either:\n"
        f"  (a) run AlpaSim yourself on your GPU host — see "
        "docs/M7_PARTICIPANT_SSM_RUNBOOK.md — which uploads to "
        f"s3://{S3_BUCKET}/{USER_M7_PREFIX}, or\n"
        "  (b) ask the workshop admin to stage the shared reference run at "
        f"s3://{SHARED_BUCKET}/{ADMIN_REF_PREFIX} (scripts/alpasim_ec2_setup.sh; "
        "see docs/ALPASIM_M7.md).\n"
        f"Expected aggregate/ (results-summary.json + metrics_results.txt/png), "
        f"rollouts/**/metrics.parquet and eval/eval.mp4 under {ref_s3}."
    )

# Inventory what came down.
print("\nReference artifacts:")
for f in sorted(glob.glob(os.path.join(LOCAL_M7, "**", "*"), recursive=True)):
    if os.path.isfile(f):
        print(f"  {os.path.relpath(f, LOCAL_M7)} ({os.path.getsize(f)/1024:.1f} KB)")

run_json = os.path.join(LOCAL_M7, "run.json")
if os.path.exists(run_json):
    with open(run_json) as f:
        run_meta = json.load(f)
    print(f"\nRun provenance: driver={run_meta.get('driver')} scene={run_meta.get('scene_id','?')[:16]}… "
          f"topology={run_meta.get('topology')} instance={run_meta.get('instance_type')}")
else:
    run_meta = {}

In [ ]:
# ============================================================
# Parse the real AlpaSim driving-score metrics
# ============================================================
# pd.read_parquet needs a parquet engine (pyarrow). SageMaker Distribution ships
# it, but on a lean kernel the import below succeeds while read_parquet later dies
# with a missing-engine ImportError — so self-heal once, quietly.
import importlib.util
if importlib.util.find_spec("pyarrow") is None and \
        importlib.util.find_spec("fastparquet") is None:
    print("(installing pyarrow for parquet reads ...)")
    subprocess.run(["python", "-m", "pip", "install", "-q", "pyarrow"], check=False)
import pandas as pd

# 1) The formatted aggregate table AlpaSim wrote (print verbatim, if present).
if os.path.exists(agg_txt):
    print("=" * 64)
    print("AlpaSim aggregate driving scores (metrics_results.txt)")
    print("=" * 64)
    with open(agg_txt) as f:
        print(f.read().strip())
    print("=" * 64)

# 2) Authoritative machine-readable aggregate: results-summary.json (one dict of
#    metric -> value, means over rollouts). This is what we key on.
summary_json = os.path.join(LOCAL_M7, "aggregate", "results-summary.json")
agg = {}
if os.path.exists(summary_json):
    with open(summary_json) as f:
        s = json.load(f)
    rows = s.get("metrics_results") if isinstance(s, dict) else s
    if isinstance(rows, list) and rows:
        agg = rows[0]
    elif isinstance(s, dict):
        agg = s
elif os.path.exists(os.path.join(LOCAL_M7, "aggregate", "metrics_results.parquet")):
    # Fallback: the wide aggregate parquet (1 row, columns = metric names).
    adf = pd.read_parquet(os.path.join(LOCAL_M7, "aggregate", "metrics_results.parquet"))
    if len(adf):
        agg = adf.iloc[0].to_dict()

# The real AlpaSim safety/quality metrics (subset that matters for the demo).
KEY_METRICS = [
    "collision_any", "collision_at_fault", "collision_rear",
    "offroad", "offroad_or_collision",
    "dist_to_gt_trajectory", "dist_traveled_m", "gt_dist_traveled_m",
    "progress_rel", "duration_frac_20s", "min_distance_to_obstacle_m",
]
print("\nGenuine driving scores (mean over rollouts):")
for m in KEY_METRICS:
    if m in agg and agg[m] is not None:
        print(f"  {m:28s}: {float(agg[m]):.3f}")

# 3) Per-rollout metrics parquet is LONG format (name/values/timestamps) — keep it
#    for the optional time-series plot in the next cell.
rollout_files = sorted(glob.glob(os.path.join(LOCAL_M7, "rollouts", "**", "metrics.parquet"),
                                 recursive=True))
rollout_df = pd.concat([pd.read_parquet(p) for p in rollout_files], ignore_index=True) \
    if rollout_files else pd.DataFrame()
if not rollout_df.empty:
    # Guard column names — different AlpaSim builds vary the parquet schema; only
    # report the counts whose columns are actually present (cell 6 guards 'valid'
    # the same way).
    n_names = rollout_df["name"].nunique() if "name" in rollout_df.columns else "?"
    n_roll = rollout_df["rollout_id"].nunique() if "rollout_id" in rollout_df.columns else "?"
    print(f"\nPer-rollout time-series: {len(rollout_df)} rows, "
          f"{n_names} metric names, {n_roll} rollout(s).")

In [ ]:
# ============================================================
# Visualize the real driving scores
# ============================================================
import matplotlib.pyplot as plt
from IPython.display import Image as IPyImage, display

# (a) AlpaSim's own aggregate summary image, if the admin uploaded it.
agg_png = os.path.join(LOCAL_M7, "aggregate", "metrics_results.png")
if os.path.exists(agg_png):
    print("AlpaSim aggregate summary (metrics_results.png):")
    display(IPyImage(filename=agg_png))
else:
    print("(aggregate/metrics_results.png not staged — plotting from data only)")

# (b) Our own plots: safety-rate bar from the aggregate dict + a time-series of
#     dist_to_gt_trajectory from the per-rollout long-format parquet.
rate_keys = [k for k in ["collision_at_fault", "collision_rear", "offroad", "collision_any"]
             if k in agg and agg[k] is not None]
have_ts = (not rollout_df.empty) and ("dist_to_gt_trajectory" in set(rollout_df.get("name", [])))
n_panels = (1 if rate_keys else 0) + (1 if have_ts else 0)

if n_panels:
    fig, axes = plt.subplots(1, n_panels, figsize=(7 * n_panels, 5), squeeze=False)
    col = 0
    if rate_keys:
        ax = axes[0][col]; col += 1
        vals = [float(agg[k]) for k in rate_keys]
        ax.bar(rate_keys, vals, color=["#e74c3c", "#e67e22", "#9b59b6", "#c0392b"][:len(rate_keys)])
        ax.set_ylabel("rate (0 = none)")
        ax.set_title("Safety violation rates (closed-loop)")
        ax.set_ylim(0, max(0.05, max(vals) * 1.3))
        for i, v in enumerate(vals):
            ax.text(i, v, f"{v:.2f}", ha="center", va="bottom", fontsize=9)
        ax.tick_params(axis="x", rotation=15)
    if have_ts:
        ax = axes[0][col]
        d = rollout_df[rollout_df["name"] == "dist_to_gt_trajectory"].copy()
        d = d[d["valid"]] if "valid" in d.columns else d
        t = (d["timestamps_us"] - d["timestamps_us"].min()) / 1e6  # seconds
        ax.plot(t, d["values"], "-", color="steelblue")
        ax.set_xlabel("time (s)")
        ax.set_ylabel("dist_to_gt_trajectory (m)")
        ax.set_title("Deviation from GT trajectory over the rollout")
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("(No plottable metrics — see the aggregate table above.)")

In [ ]:
# ============================================================
# Play the real closed-loop evaluation video
# ============================================================
# This is the actual AlpaSim rollout of the Alpamayo policy driving the NuRec
# scene (with the Chain-of-Causation reasoning overlay, tying back to M6).
from IPython.display import Video, display

eval_mp4 = None
for cand in [os.path.join(LOCAL_M7, "eval", "eval.mp4")] + sorted(
        glob.glob(os.path.join(LOCAL_M7, "**", "*.mp4"), recursive=True)):
    if os.path.exists(cand):
        eval_mp4 = cand
        break

if eval_mp4:
    size_mb = os.path.getsize(eval_mp4) / 1e6
    print(f"Closed-loop eval video: {os.path.relpath(eval_mp4, LOCAL_M7)} ({size_mb:.1f} MB)")
    try:
        display(Video(eval_mp4, embed=True, width=720))
    except Exception as e:
        print(f"(inline video preview skipped: {e} — file is present at {eval_mp4})")
else:
    print("(No eval video in the reference bundle — metrics above are the primary result.)")

In [ ]:
# ============================================================
# Cost — honest framing
# ============================================================
print("=" * 60)
print("COST — M7 AlpaSim Closed-Loop")
print("=" * 60)
print("This notebook ran on CPU (ml.t3.medium) — negligible cost. It only")
print("downloaded and visualized results.")
print()
print("The underlying AlpaSim closed-loop evaluation is compute-heavy (a")
print("dedicated >=40 GB GPU + Docker microservices + the NuRec renderer).")
if RESULT_SOURCE == "your own EC2 run":
    _inst = run_meta.get("instance_type") if run_meta else None
    _host = _inst if _inst and _inst != "unknown" else "a GPU host (e.g. g6e.12xlarge, ~$10.5/hr)"
    print(f"You ran it yourself on a pre-provisioned host ({_host})")
    print("reached over SSM — see docs/M7_PARTICIPANT_SSM_RUNBOOK.md.")
    print("Make sure you told the admin you're DONE so they terminate that host.")
else:
    _inst = run_meta.get("instance_type") if run_meta else None
    _host = _inst if _inst and _inst != "unknown" else "a GPU EC2 host (e.g. g6e.12xlarge)"
    print(f"The workshop admin ran it once on {_host} (see docs/ALPASIM_M7.md);")
    print("that one-time cost (~$30) is shared across all participants —")
    print("you paid nothing for it.")
if run_meta:
    print()
    print(f"Reference run: instance={run_meta.get('instance_type','?')} "
          f"topology={run_meta.get('topology','?')} renderer={run_meta.get('renderer_image','?')}")
print("=" * 60)

In [ ]:
# ============================================================
# Validation + pipeline end
# ============================================================
# Validates that the genuine reference artifacts loaded and parsed — NOT that
# this notebook simulated anything (the GPU host did, yours or the admin's).
checks = [
    ("aggregate driving scores parsed", bool(agg)),
    ("collision_at_fault present", "collision_at_fault" in agg),
    ("per-rollout time-series present", (not rollout_df.empty)),
    ("eval video present", eval_mp4 is not None),
]
print("Reference-eval validation:")
for name, passed in checks:
    print(f"    {name}: {'OK' if passed else 'MISSING'}")
# The aggregate scores are the required result; time-series/video are nice-to-have.
required_ok = checks[0][1] and checks[1][1]
print(f"  Status: {'PASS — reference eval loaded and visualized' if required_ok else 'FAIL — reference eval incomplete'}")

# Headline safety result, stated plainly.
if agg:
    coll = agg.get("collision_at_fault")
    off = agg.get("offroad")
    prog = agg.get("progress_rel")
    if coll is not None:
        verdict = "no at-fault collisions" if float(coll) == 0 else f"at-fault collision rate {float(coll):.2f}"
        extra = []
        if off is not None:
            extra.append("no off-road" if float(off) == 0 else f"offroad {float(off):.2f}")
        if prog is not None:
            extra.append(f"route progress {float(prog):.2f}")
        print(f"\nHeadline: Alpamayo drove closed-loop with {verdict}" +
              (f" ({', '.join(extra)})" if extra else "") + ".")

# Where the M7 result you just visualized came from.
m7_out = (f"s3://{S3_BUCKET}/{USER_M7_PREFIX} (your own run)"
          if RESULT_SOURCE == "your own EC2 run"
          else f"s3://{SHARED_BUCKET}/{ADMIN_REF_PREFIX} (admin reference)")

print("\n" + "=" * 60)
print("PIPELINE COMPLETE")
print("=" * 60)
print("The AV 3.0 Blueprint Lab pipeline has completed.")
print("\nSummary of outputs:")
print(f"  M4: Weather-augmented clips     → users/{PROFILE}/m4/")
print(f"  M5: Synthetic scenarios          → users/{PROFILE}/m5/")
print(f"  M6: Alpamayo open-loop (minADE)  → users/{PROFILE}/m6/")
print(f"  M7: AlpaSim closed-loop eval     → {m7_out}")
print("\nM6 → M7: the same Alpamayo-1.5-10B checkpoint, evaluated open-loop (M6)")
print("then closed-loop in AlpaSim (M7).")

In [ ]:
"""Mark this module complete on the participant dashboard (best-effort, non-fatal)."""
import sys
from pathlib import Path
for _b in (Path.cwd(), Path.cwd().parent, Path.home()):
    _cand = _b / "scripts" / "av30_progress.py"
    if _cand.exists():
        sys.path.insert(0, str(_b / "scripts"))
        break
try:
    from av30_progress import mark_complete
    if required_ok:
        mark_complete("m07-alpasim")
    else:
        print("[progress] reference eval incomplete — not marking M7 complete.")
except Exception as _e:
    print(f"[progress] helper unavailable ({_e}); skipping — module still complete.")